# Step 6: Self-Supervised Temporal Pretraining

**Objective**: Pretrain the shared encoder using unlabeled temporal cine frames.

**SSL Tasks**:
1. **Masked Reconstruction**: Mask random patches, reconstruct from encoder features
2. **Temporal Consistency**: Minimize feature distance between adjacent cardiac frames

**Key**: Uses ALL temporal frames (not just labeled ED/ES), maximizing data usage.

**Loss**: L_ssl = L_recon + λ_temporal × L_temporal (λ_temporal = 0.1)

In [ ]:
import sys
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import ACDCTemporalDataset
from src.ssl import SSLModel, SSLTrainer
from src.train import set_seed, get_device, get_adaptive_batch_size
from src.encoder import count_parameters

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')

SEED = 42
DEVICE = get_device('auto')
SSL_BATCH_SIZE = get_adaptive_batch_size(DEVICE, default=16)

set_seed(SEED)
print(f"SSL Batch size: {SSL_BATCH_SIZE}")

## 6.1 Setup Temporal Dataset

In [ ]:
# Load temporal dataset (all frames, train split)
temporal_dataset = ACDCTemporalDataset(
    processed_dir=PROCESSED_DIR,
    split_file=os.path.join(SPLITS_DIR, 'train.json'),
)

print(f"Temporal pairs available: {len(temporal_dataset)}")

temporal_loader = DataLoader(
    temporal_dataset,
    batch_size=SSL_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)

# Verify a sample
sample = temporal_dataset[0]
print(f"\nSample:")
print(f"  frame_t shape: {sample['frame_t'].shape}")
print(f"  frame_t1 shape: {sample['frame_t1'].shape}")
print(f"  patient_id: {sample['patient_id']}")
print(f"  frame_idx_t: {sample['frame_idx_t']}, frame_idx_t1: {sample['frame_idx_t1']}")

In [ ]:
# Visualize temporal pairs
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for i in range(5):
    idx = i * (len(temporal_dataset) // 5)
    sample = temporal_dataset[idx]
    
    axes[0, i].imshow(sample['frame_t'].squeeze(), cmap='gray')
    axes[0, i].set_title(f"{sample['patient_id']}\nFrame {sample['frame_idx_t']}")
    axes[0, i].axis('off')
    
    axes[1, i].imshow(sample['frame_t1'].squeeze(), cmap='gray')
    axes[1, i].set_title(f"Frame {sample['frame_idx_t1']}")
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Frame t', fontsize=12)
axes[1, 0].set_ylabel('Frame t+1', fontsize=12)
fig.suptitle('Temporal Frame Pairs for SSL Pretraining', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 6.2 Build SSL Model and Train

In [ ]:
set_seed(SEED)

# Build SSL model
ssl_model = SSLModel(
    in_channels=1,
    encoder_channels=[32, 64, 128, 256],
    proj_dim=128,
    mask_patch_size=16,
    mask_ratio=0.5,
    dropout=0.1,
)

total_params = count_parameters(ssl_model)
encoder_params = count_parameters(ssl_model.encoder)
print(f"SSL model parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"Encoder parameters:   {encoder_params:,} ({encoder_params/1e6:.2f}M) — transferred to segmentation")
print(f"SSL-only parameters:  {total_params - encoder_params:,} — discarded after pretraining")

# Verify forward pass
dummy_t = torch.randn(2, 1, 256, 256)
dummy_t1 = torch.randn(2, 1, 256, 256)
with torch.no_grad():
    results = ssl_model(dummy_t, dummy_t1)
print(f"\nForward pass outputs:")
for k, v in results.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")
print("✓ SSL model forward pass verified")

In [ ]:
# SSL Optimizer and Trainer
ssl_optimizer = torch.optim.AdamW(ssl_model.parameters(), lr=1e-4, weight_decay=1e-5)
ssl_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ssl_optimizer, T_max=100, eta_min=1e-6)

ssl_trainer = SSLTrainer(
    model=ssl_model,
    optimizer=ssl_optimizer,
    device=DEVICE,
    recon_weight=1.0,
    temporal_weight=0.1,
    mixed_precision=True,
    scheduler=ssl_scheduler,
)

# Train
ssl_history = ssl_trainer.train(
    dataloader=temporal_loader,
    n_epochs=100,
    save_dir=CHECKPOINT_DIR,
    save_interval=25,
)

## 6.3 Visualize SSL Results

In [ ]:
# Plot SSL training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ssl_history['total_loss'], color='steelblue', label='Total')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total SSL Loss')
axes[0].legend()

axes[1].plot(ssl_history['recon_loss'], color='coral', label='Reconstruction')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Reconstruction Loss')
axes[1].legend()

axes[2].plot(ssl_history['temporal_loss'], color='mediumpurple', label='Temporal')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].set_title('Temporal Consistency Loss')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'ssl_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize reconstructions
ssl_model.eval()
ssl_model = ssl_model.to(DEVICE)

fig, axes = plt.subplots(3, 5, figsize=(18, 10))

for i in range(5):
    idx = i * (len(temporal_dataset) // 5)
    sample = temporal_dataset[idx]
    frame = sample['frame_t'].unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        results = ssl_model(frame)
    
    orig = frame.squeeze().cpu().numpy()
    masked = (frame * (1 - results['mask'])).squeeze().cpu().numpy()
    recon = results['reconstructed'].squeeze().cpu().numpy()
    
    axes[0, i].imshow(orig, cmap='gray')
    axes[0, i].set_title(f'Original')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(masked, cmap='gray')
    axes[1, i].set_title(f'Masked (50%)')
    axes[1, i].axis('off')
    
    axes[2, i].imshow(recon, cmap='gray')
    axes[2, i].set_title(f'Reconstructed')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Masked', fontsize=12)
axes[2, 0].set_ylabel('Reconstructed', fontsize=12)
fig.suptitle('SSL Masked Reconstruction Results', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'ssl_reconstructions.png'), dpi=150, bbox_inches='tight')
plt.show()

# Sanity checks
final_recon = ssl_history['recon_loss'][-1]
final_temp = ssl_history['temporal_loss'][-1]
print(f"\nFinal reconstruction loss: {final_recon:.4f}")
print(f"Final temporal loss: {final_temp:.4f}")
print(f"Loss reduction: {ssl_history['recon_loss'][0]:.4f} → {final_recon:.4f}")

if final_recon < ssl_history['recon_loss'][0]:
    print("✓ Reconstruction loss decreased — encoder is learning")
else:
    print("⚠ Reconstruction loss did not decrease — check training")

print(f"\nEncoder saved to: {os.path.join(CHECKPOINT_DIR, 'ssl_encoder_final.pth')}")
print("\n=== Step 6: SSL Pretraining COMPLETE ===")